# Machine Learning
Pasar de serie temporal a problema supervisado, variables retardadas, predictores ambientales y validación temporal.

1. Problema científico: ¿Qué pregunta oceanográfica puede abordar?
Las variables oceanográficas están relacionadas entre sí mediante procesos físicos, químicos y biológicos. Los métodos de Machine Learning permiten aprender relaciones estadísticas entre una variable de interés y un conjunto de variables explicativas a partir de observaciones históricas. A diferencia de los modelos estadísticos clásicos de series temporales, estos métodos no necesitan definir explícitamente una estructura de tendencia, estacionalidad o dependencia temporal, aunque pueden incorporar información temporal entre sus variables de entrada.

2. Datos necesarios: ¿Qué variables, resolución temporal y longitud de serie requiere?
En datos oceanográficos, las observaciones de una misma campaña, estación o periodo pueden estar fuertemente relacionadas. Por ello, la división entre entrenamiento y evaluación debe tener en cuenta la estructura temporal y espacial de los datos.

3. Fundamento: ¿Qué variables, resolución temporal y longitud de serie requiere?
En datos oceanográficos, las observaciones de una misma campaña, estación o periodo pueden estar fuertemente relacionadas. Por ello, la división entre entrenamiento y evaluación debe tener en cuenta la estructura temporal y espacial de los datos.

## Ejemplo reproducible: ¿Cómo se aplica a datos reales o a un caso didáctico?

En este notebook se utiliza Random Forest (RF), un algoritmo de aprendizaje supervisado, para explorar la predicción de una variable oceanográfica a partir de otras variables ambientales. Random Forest no es, por sí mismo, un modelo de series temporales. Es un algoritmo de aprendizaje supervisado que puede utilizarse para predecir una variable oceanográfica a partir de otras variables, incluidas las observaciones pasadas, si se diseñan las entradas de esa manera.

## Ejemplo propuesto: Random Forest (RF)

**Objetivo** A. Estimar clorofila-a a partir de las condiciones hidrográficas, estimar clorofila a partir de variables medidas en la misma observación. Es un problema de regresión supervisada, pero no necesariamente una predicción futura. B. Predecir clorofila-a en un momento futuro, predecir clorofila futura a partir de información disponible en el momento de la predicción, como medidas pasadas, estacionalidad y variables ambientales conocidas o pronosticadas.

**Datos:** Fluorescencia superficie mensual (E2GI) + temperatura superficie mensual (E2GI).

**Ventajas** El océano real es altamente no lineal y las variables interactúan de formas complejas. Los métodos de ML encuentran patrones complejos e interacciones entre muchas variables.

**Desventajas** Es una "caja negra"; no respeta las leyes de la física por sí solo. Las relaciones identificadas por el modelo no deben interpretarse automáticamente como mecanismos físicos o biológicos, ni como relaciones causales.

plot *clorofila superficie*, diferenciando periodo de entrenamiento y periodo de test.

In [ ]:
1. Construir la matriz de features para el caudal del Genil (lags + rolling + calendario + Fourier + lluvia retardada)
2. Implementar baselines: persistencia, persistencia estacional, climatología.
3. Ajustar una regresión Ridge con `skforecast` y comparar.

In [ ]:
caudal = ud.cargar_caudal_genil(source="CEDEX")
lluvia = ud.cargar_lluvia_genil_diaria(fecha_inicio="2010-01-01", fecha_fin="2020-12-31")

# Periodo 2011-2020 (10 años con cobertura diaria continua del SAIH).
# Interpolamos los pocos gaps cortos que quedan (≤7 días) para tener freq='D' limpia.
    df = pd.DataFrame({"caudal": caudal, "lluvia": lluvia}).loc["2011\":"2020"]
    df = df.asfreq("D").interpolate("linear", limit=7).dropna()
    print(f\"Días: {len(df):,}   Freq: {df.index.freq}   NaN: {df.isna().sum().sum()}")
    print(df.describe().round(2))


In [ ]:
Split temporal

In [ ]:
split = pd.Timestamp(\"2018-01-01\")\n",
    "train = feat.loc[:split].iloc[:-1]\n",
    "test = feat.loc[split:]\n"
                     y_train, y_test = train[\"y\"], test[\"y\"]\n",
    "X_train = train.drop(columns=\"y\")\n",
    "X_test = test.drop(columns=\"y\")\n",
    "print(f\"Train: {len(train):,} obs   Test: {len(test):,} obs   Features: {X_train.shape[1]}\")"


4. Validación: ¿Cómo sabemos si predice razonablemente bien?

La evaluación debe determinar si el modelo predice adecuadamente observaciones que no ha utilizado durante su entrenamiento. Un buen ajuste sobre los datos de entrenamiento no garantiza que el modelo generalice a nuevos datos.
Los datos suelen dividirse en un conjunto de entrenamiento (train) y otro de prueba (test). El modelo se ajusta con el primero y sus predicciones se comparan con los valores observados en el segundo. Cuando existen observaciones temporal o espacialmente relacionadas, no se barajan.
Si se pretende estimar el rendimiento en condiciones ambientales nuevas, debe comprobarse si esas condiciones están representadas en el entrenamiento (dataset shift).

Entre las métricas de regresión más habituales se encuentran:

MAE (Mean Absolute Error): error absoluto medio, expresado en las unidades de la variable objetivo.

RMSE (Root Mean Squared Error): raíz del error cuadrático medio, que penaliza especialmente los errores grandes.

R² (coeficiente de determinación): compara el error del modelo con la variabilidad de los valores observados respecto a su media.

MAE, RMSE, MAPE, sMAPE: ventajas y trampas.
Evaluación específica de eventos extremos?
Métricas y evaluación: Compara MAE, RMSE y, si resulta útil, \(R^2\). Analiza qué modelo predice mejor los cambios de concentración y los episodios de valores elevados.

5. Limitaciones: ¿Cuándo falla y qué conclusiones no permite extraer?
ML es útil para identificar relaciones predictivas complejas entre variables oceanográficas, pero presenta limitaciones que deben considerarse al interpretar sus resultados.

El modelo puede tener dificultades cuando se aplica a condiciones ambientales fuera del rango representado en los datos de entrenamiento. Del mismo modeo, un muestreo sesgado, escaso o poco representativo puede limitar la capacidad de generalización.

Aunque las importancias de variables y otras herramientas de interpretación permiten examinar el comportamiento del modelo, no proporcionan por sí mismas una explicación mecanística del proceso.

El modelo puede no reproducir adecuadamente episodios raros o valores fuera del rango observado durante el entrenamiento (eventos extremos)

6. Información adicional ¿Dónde puede profundizar quien quiera utilizarlo en investigación?

https://scikit-learn.org/stable/modules/ensemble.html

## A mayores: Redes neuronales para series temporales
Las redes neuronales artificiales constituyen otra familia de métodos de aprendizaje automático que puede aplicarse a la predicción de variables oceanográficas. Aprenden representaciones y relaciones entre variables mediante capas de unidades de procesamiento conectadas.

Tenemos compañeros que las han utilizado ampliamente en tareas de clasificación de imágenes. Por lo que yo se, también existen arquitecturas diseñadas para trabajar con datos secuenciales y series temporales, como las redes recurrentes (RNN), las redes LSTM (Long Short-Term Memory), las redes convolucionales temporales y los Transformers. Estas arquitecturas pueden utilizarse para aprender patrones temporales y relaciones entre múltiples variables ambientales.Este notebook no desarrolla estas técnicas por que no tengo ni idea, pero las dejo aquí mencionadas como una línea de trabajo complementaria para futuras ampliaciones de OceanQAst.